In [56]:
import io
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import files

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ==========================================
# ALTERNATIVE DATA LOADING METHODS
# ==========================================

METHOD = "Local Path" #@param ["Local Path", "Direct URL", "Colab Upload Widget"]
FILE_PATH_OR_URL = "/content/synthetic_traffic_benchmark_200k.csv" #@param {type:"string"}

if METHOD == "Local Path":
    print(f"Attempting to load file from local path: {FILE_PATH_OR_URL}")
    if os.path.exists(FILE_PATH_OR_URL):
        df = pd.read_csv(FILE_PATH_OR_URL)
        print("\nDataset loaded successfully from local path!")
    else:
        print(f"\n[ERROR]: File not found at '{FILE_PATH_OR_URL}'.")
        print("Please upload your file using the folder icon on the left sidebar, or change the METHOD.")

elif METHOD == "Direct URL":
    print(f"Attempting to download and load file from URL: {FILE_PATH_OR_URL}")
    try:
        local_filename = "downloaded_dataset.csv"
        urllib.request.urlretrieve(FILE_PATH_OR_URL, local_filename)
        df = pd.read_csv(local_filename)
        print("\nDataset downloaded and loaded successfully!")
    except Exception as e:
        print(f"\n[ERROR]: Failed to download from URL. Details: {e}")

elif METHOD == "Colab Upload Widget":
    print("Please click the button below to upload your dataset CSV file:")
    uploaded = files.upload()
    if not uploaded:
        print("\n[ERROR]: No file was selected.")
    else:
        csv_filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[csv_filename]))
        print(f"\nDataset loaded successfully from uploaded file: {csv_filename}!")

if 'df' in globals():
    print("Dataset shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


Attempting to load file from local path: /content/synthetic_traffic_benchmark_200k.csv

[ERROR]: File not found at '/content/synthetic_traffic_benchmark_200k.csv'.
Please upload your file using the folder icon on the left sidebar, or change the METHOD.
Dataset shape: (0, 250)
Columns: ['{"metadata":{"kernelspec":{"language":"python"', 'display_name:"Python 3"', 'name:"python3"}', 'language_info:{"name":"python"', 'version:"3.12.13"', 'mimetype:"text/x-python"', 'codemirror_mode:{"name":"ipython"', 'version:3}', 'pygments_lexer:"ipython3"', 'nbconvert_exporter:"python"', 'file_extension:".py"}', 'kaggle:{"accelerator":"none"', 'dataSources:[{"sourceType":"datasetVersion"', 'sourceId:19459695}]', 'isInternetEnabled:true', 'language:"python"', 'sourceType:"notebook"', 'isGpuEnabled:false}}', 'nbformat_minor:4', 'nbformat:4', 'cells:[{"cell_type":"markdown"', 'source:"## ⚙️ 1. Environment Setup & Data Ingestion\\n\\nWe begin by setting up the **Python environment** and importing the essent

,"{""metadata"":{""kernelspec"":{""language"":""python""","display_name:""Python 3""","name:""python3""}","language_info:{""name"":""python""","version:""3.12.13""","mimetype:""text/x-python""","codemirror_mode:{""name"":""ipython""",version:3},"pygments_lexer:""ipython3""","nbconvert_exporter:""python""",...,"output_type:""stream""}","{""output_type"":""display_data""","data:{""text/plain"":""<Figure size 800x600 with 1 Axes>""","image/png:""...\n""}",metadata:{}}.6,"{""output_type"":""display_data"".1","data:{""text/plain"":""<Figure size 1000x600 with 1 Axes>""","image/png:""...\n""}",metadata:{}}].1,execution_count:23}]}


In [ ]:
# STEP 2: DATA CLEANING AND CATEGORICAL ENCODING

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Display actual column names
print("Available columns:")
print(df.columns.tolist())

# Remove spaces from column names
df.columns = df.columns.str.strip()

# Automatically identify timestamp column
time_candidates = [
    "Timestamp", "timestamp", "DateTime", "Datetime",
    "Date_Time", "date_time", "Date", "Time"
]

time_col = next(
    (c for c in time_candidates if c in df.columns),
    None
)

# Automatically identify congestion target column
target_candidates = [
    "Congestion_Index", "congestion_index",
    "Traffic_Congestion", "Traffic Congestion",
    "Congestion", "congestion",
    "Traffic_Density", "traffic_density",
    "Traffic_Speed", "traffic_speed"
]

target_col = next(
    (c for c in target_candidates if c in df.columns),
    None
)

# Stop safely if columns cannot be identified
if time_col is None:
    print("ERROR: Timestamp column not found.")
    print("Please check the column names printed above.")

elif target_col is None:
    print("ERROR: Congestion target column not found.")
    print("Please check the column names printed above.")

else:
    print("\nTime column:", time_col)
    print("Target column:", target_col)

    # Convert timestamp
    df[time_col] = pd.to_datetime(
        df[time_col], errors="coerce"
    )

    # Remove rows with invalid timestamps
    df = df.dropna(subset=[time_col]).copy()

    # Convert target into numerical values
    df[target_col] = pd.to_numeric(
        df[target_col], errors="coerce"
    )

    # Fill missing target values
    df[target_col] = df[target_col].interpolate(
        limit_direction="both"
    )

    # Remove rows if target is still missing
    df = df.dropna(subset=[target_col]).copy()

    # Encode categorical columns
    categorical_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in categorical_cols:
        if col == time_col:
            continue

        df[col] = df[col].fillna("Unknown").astype(str)

        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col])

    # Fill missing numerical values
    numerical_cols = df.select_dtypes(
        include=np.number
    ).columns

    for col in numerical_cols:
        median_value = df[col].median()
        df[col] = df[col].fillna(
            median_value if pd.notna(median_value) else 0
        )

    # Sort by timestamp
    df = df.sort_values(time_col).reset_index(drop=True)

    print("\nData preprocessing completed!")
    print("Dataset shape:", df.shape)
    print("Remaining missing values:", df.isnull().sum().sum())
    display(df.head())

Available columns:
['{"metadata":{"kernelspec":{"language":"python"', 'display_name:"Python 3"', 'name:"python3"}', 'language_info:{"name":"python"', 'version:"3.12.13"', 'mimetype:"text/x-python"', 'codemirror_mode:{"name":"ipython"', 'version:3}', 'pygments_lexer:"ipython3"', 'nbconvert_exporter:"python"', 'file_extension:".py"}', 'kaggle:{"accelerator":"none"', 'dataSources:[{"sourceType":"datasetVersion"', 'sourceId:19459695}]', 'isInternetEnabled:true', 'language:"python"', 'sourceType:"notebook"', 'isGpuEnabled:false}}', 'nbformat_minor:4', 'nbformat:4', 'cells:[{"cell_type":"markdown"', 'source:"## ⚙️ 1. Environment Setup & Data Ingestion\\n\\nWe begin by setting up the **Python environment** and importing the essential libraries required for data manipulation', ' visualization', ' and machine learning.\\n\\nNext', ' we load the **training and testing datasets** using `pandas`', ' ensuring the competition data is correctly imported and ready for analysis.\\n\\nThis step establis

In [ ]:
print(df.columns.tolist())

['{"metadata":{"kernelspec":{"language":"python"', 'display_name:"Python 3"', 'name:"python3"}', 'language_info:{"name":"python"', 'version:"3.12.13"', 'mimetype:"text/x-python"', 'codemirror_mode:{"name":"ipython"', 'version:3}', 'pygments_lexer:"ipython3"', 'nbconvert_exporter:"python"', 'file_extension:".py"}', 'kaggle:{"accelerator":"none"', 'dataSources:[{"sourceType":"datasetVersion"', 'sourceId:19459695}]', 'isInternetEnabled:true', 'language:"python"', 'sourceType:"notebook"', 'isGpuEnabled:false}}', 'nbformat_minor:4', 'nbformat:4', 'cells:[{"cell_type":"markdown"', 'source:"## ⚙️ 1. Environment Setup & Data Ingestion\\n\\nWe begin by setting up the **Python environment** and importing the essential libraries required for data manipulation', 'visualization', 'and machine learning.\\n\\nNext', 'we load the **training and testing datasets** using `pandas`', 'ensuring the competition data is correctly imported and ready for analysis.\\n\\nThis step establishes a clean and reprodu

In [58]:
# STEP 3: HOURLY GROUPING
import sys
import numpy as np
import pandas as pd

# Safely verify that time_col and target_col were successfully identified
if 'time_col' not in globals() or time_col is None:
    print("[ERROR]: 'time_col' is None. We cannot proceed with hourly grouping.", file=sys.stderr)
    print("This is because a Jupyter Notebook (.ipynb) was uploaded in Step 1 instead of your dataset CSV file.", file=sys.stderr)
    print("Please upload the correct dataset CSV file in Step 1 and re-run the cells sequentially.", file=sys.stderr)
else:
    df["Hour"] = df[time_col].dt.floor("h")

    # Exclude timestamp and date from the input features
    numeric_cols = [
        c for c in df.select_dtypes(include=np.number).columns
        if c != target_col or c in df.columns
    ]

    hourly_df = df.groupby("Hour")[numeric_cols].mean()
    hourly_df = hourly_df.sort_index()

    # Create a continuous hourly time index
    all_hours = pd.date_range(
        start=hourly_df.index.min(),
        end=hourly_df.index.max(),
        freq="h"
    )

    hourly_df = hourly_df.reindex(all_hours)
    hourly_df.index.name = "Timestamp"

    # Fill gaps in hourly data
    hourly_df = hourly_df.interpolate(
        method="time",
        limit_direction="both"
    )

    hourly_df = hourly_df.ffill().bfill().fillna(0)

    print("Hourly grouping completed!")
    print("Hourly records:", len(hourly_df))
    display(hourly_df.head())

[ERROR]: 'time_col' is None. We cannot proceed with hourly grouping.
This is because a Jupyter Notebook (.ipynb) was uploaded in Step 1 instead of your dataset CSV file.
Please upload the correct dataset CSV file in Step 1 and re-run the cells sequentially.


In [60]:
# STEP 4: NORMALIZATION AND SEQUENCE GENERATION

import sys
import numpy as np
from sklearn.preprocessing import StandardScaler

LOOKBACK = 12

# Safely verify that hourly_df and target_col exist
if 'hourly_df' not in globals() or 'target_col' not in globals() or target_col is None:
    print("[ERROR]: 'hourly_df' or 'target_col' is not defined. We cannot proceed with normalization.", file=sys.stderr)
    print("Please upload the correct dataset CSV file in Step 1, run the data cleaning, and hourly grouping cells before running this cell.", file=sys.stderr)
else:
    if target_col not in hourly_df.columns:
        raise ValueError("Congestion_Index is missing after hourly grouping.")

    if len(hourly_df) < 30:
        raise ValueError(
            "Too few hourly records. Use a dataset covering more hours."
        )

    split_index = int(len(hourly_df) * 0.80)

    # Fit scalers only on the training period
    feature_scaler = StandardScaler()
    feature_scaler.fit(hourly_df.iloc[:split_index])

    scaled_features = feature_scaler.transform(hourly_df)

    target_scaler = StandardScaler()
    target_scaler.fit(
        hourly_df[[target_col]].iloc[:split_index]
    )

    scaled_target = target_scaler.transform(
        hourly_df[[target_col]]
    ).ravel()

    X, y, positions = [], [], []

    for i in range(LOOKBACK, len(hourly_df)):
        X.append(scaled_features[i-LOOKBACK:i])
        y.append(scaled_target[i])
        positions.append(i)

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    positions = np.asarray(positions)

    train_mask = positions < split_index
    test_mask = positions >= split_index

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    test_positions = positions[test_mask]

    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("Previous hours used:", LOOKBACK)

[ERROR]: 'hourly_df' or 'target_col' is not defined. We cannot proceed with normalization.
Please upload the correct dataset CSV file in Step 1, run the data cleaning, and hourly grouping cells before running this cell.


In [62]:
# STEP 5: LSTM MODEL

import sys
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

tf.keras.utils.set_random_seed(42)

# Safely verify that X_train is ready before defining the network structure
if 'X_train' not in globals() or X_train is None:
    print("[ERROR]: 'X_train' is not defined. We cannot instantiate the model inputs.", file=sys.stderr)
    print("Please complete Step 4 (Normalization & Sequence Generation) successfully before running this cell.", file=sys.stderr)
else:
    model = Sequential([
        tf.keras.Input(shape=(X_train.shape[1], X_train.shape[2])),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    model.summary()

[ERROR]: 'X_train' is not defined. We cannot instantiate the model inputs.
Please complete Step 4 (Normalization & Sequence Generation) successfully before running this cell.


In [64]:
# STEP 6: MODEL TRAINING

import sys
from tensorflow.keras.callbacks import EarlyStopping

# Safely verify that model and training data exist
if 'model' not in globals() or 'X_train' not in globals() or 'y_train' not in globals():
    print("[ERROR]: 'model' or 'X_train' is not defined. We cannot train the network.", file=sys.stderr)
    print("Please upload the correct dataset CSV in Step 1, then run all steps (1 to 5) sequentially before running this cell.", file=sys.stderr)
else:
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.15,
        callbacks=[early_stop],
        shuffle=False,
        verbose=1
    )

    print("Model training completed!")

[ERROR]: 'model' or 'X_train' is not defined. We cannot train the network.
Please upload the correct dataset CSV in Step 1, then run all steps (1 to 5) sequentially before running this cell.


In [65]:
# STEP 7: VISUALIZATION PERFORMANCE

import sys

# Safely check if 'history' exists in the environment
if 'history' not in globals() or history is None:
    print("[ERROR]: 'history' is not defined because the model training step (Step 6) has not run successfully.", file=sys.stderr)
    print("Please upload the correct CSV dataset in Step 1, then run all cells sequentially to train the model.", file=sys.stderr)
else:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 4))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")

    plt.title("LSTM Training Performance")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

[ERROR]: 'history' is not defined because the model training step (Step 6) has not run successfully.
Please upload the correct CSV dataset in Step 1, then run all cells sequentially to train the model.
